# KernelBench-RL: Free Colab T4 Reproducibility Notebook (7B QLoRA)

This notebook reproduces the bounded **7B QLoRA (NF4, fp16)** GPU kernel synthesis run and held-out evaluation on a **free Google Colab Tesla T4 (`sm75`, 14.56 GB VRAM)**.

### Important Scope & Reproducibility Notes
- **Zero cost / zero credentials:** Uses only public repositories, public Hugging Face weights (`token=False`), and local Colab disk (`/content/t4-results`). No Colab upgrades, Drive mounts, API keys, or paid endpoints are used.
- **Free GPU quotas are not guaranteed:** Free Colab T4 sessions may disconnect, throttle, or hit daily GPU limits before completion. Checkpoints are saved every 2 steps so you can resume explicitly if needed.
- **7B QLoRA, not full-parameter 7B:** Trains **5,046,272 LoRA adapter parameters** (rank 8, alpha 16 on `q_proj`, `k_proj`, `v_proj`, `o_proj`) over a frozen 4-bit NF4 base model (`Qwen/Qwen2.5-Coder-7B-Instruct`).
- **No quality or speedup claims:** Held-out evaluation covers a small configured set (`N=3`) and a bounded 8-step run. Negative or neutral results are valid outcomes.
- **Safe default (`RUN_FULL_TRAINING_AND_EVAL = False`):** Running the notebook as-is performs only the isolated venv setup, GPU check, and real Triton worker preflight on `rmsnorm_2048`, then exits cleanly **before** downloading the ~14 GB model or starting training. Flip `RUN_FULL_TRAINING_AND_EVAL = True` explicitly when you are ready for the full run.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path
from typing import Any, Sequence

# --- Safe Defaults & Explicit Opt-In ---
# Keep False for smoke preflight only (verifies T4 + Triton worker, exits before 14 GB download).
RUN_FULL_TRAINING_AND_EVAL: bool = False

# Optional explicit checkpoint directory to resume from (leave empty for a fresh run).
RESUME_FROM_CHECKPOINT: str = ""

# --- Pinned Public Repository & Model ---
REPO_URL: str = "https://github.com/Eljo-M/kernelbench-rl.git"
REPO_DIR: Path = Path("/content/kernelbench-rl")
REPO_COMMIT: str = "e72c42e794643ed4c198937e86427c3640b977c5"
REPO_REF_OVERRIDE: str = ""  # Optional branch/tag/commit override; empty uses REPO_COMMIT.

MODEL_ID: str = "Qwen/Qwen2.5-Coder-7B-Instruct"
MODEL_REVISION: str = "c03e6d358207e414f1eca0bb1891e29f1db0e242"
EXPECTED_QLORA_TRAINABLE_PARAMS: int = 5_046_272

# --- Isolated Environment & Output Paths ---
VENV_DIR: Path = Path("/content/kernelbench-venv")
VENV_PYTHON: Path = VENV_DIR / "bin" / "python"
RESULTS_ROOT: Path = Path("/content/t4-results")
RUN_NAME: str = "t4_7b_qlora_run1"
RUN_DIR: Path = RESULTS_ROOT / RUN_NAME
PYTORCH_CU124_INDEX: str = "https://download.pytorch.org/whl/cu124"

# --- Bounded T4 QLoRA Training Parameters ---
MAX_NEW_TOKENS: int = 1024
MAX_STEPS: int = 8
GROUP_SIZE: int = 2
CHECKPOINT_EVERY: int = 2
WARMUP_ITERS: int = 5
BENCH_ITERS: int = 30
WORKER_TIMEOUT_SECS: float = 120.0


def build_subprocess_env(extra: dict[str, str] | None = None) -> dict[str, str]:
    """Preserve host environment including LD_LIBRARY_PATH without mutating host drivers."""
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["HF_HUB_DISABLE_TELEMETRY"] = "1"
    if extra:
        env.update(extra)
    return env


def run_cmd(
    argv: Sequence[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    echo: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run an argv list with check=True and captured stdout/stderr (no shell interpolation)."""
    proc = subprocess.run(
        list(argv),
        cwd=str(cwd) if cwd is not None else None,
        env=build_subprocess_env() if env is None else env,
        check=True,
        capture_output=True,
        text=True,
    )
    if echo and proc.stdout.strip():
        print(proc.stdout.strip())
    if echo and proc.stderr.strip():
        print(proc.stderr.strip(), file=sys.stderr)
    return proc


print(f"Host Python: {sys.version.split()[0]}")
print(f"Mode: {'FULL TRAINING + EVAL' if RUN_FULL_TRAINING_AND_EVAL else 'SMOKE PREFLIGHT ONLY'}")
print(f"Target run dir: {RUN_DIR}")
print(f"LD_LIBRARY_PATH preserved: {bool(os.environ.get('LD_LIBRARY_PATH'))}")

## Step 1: Clone Repository (If Needed) & Checkout Pinned Commit

Clones the public repository only when `/content/kernelbench-rl` does not already exist, then checks out the pinned commit (`REPO_COMMIT`, or `REPO_REF_OVERRIDE` if explicitly set).

In [ ]:
if not REPO_DIR.exists():
    run_cmd(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print(f"Repository already exists at {REPO_DIR}; skipping clone.")

run_cmd(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR, echo=False)
target_ref = REPO_REF_OVERRIDE.strip() or REPO_COMMIT
run_cmd(["git", "checkout", "--detach", target_ref], cwd=REPO_DIR)

RESOLVED_COMMIT = run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, echo=False).stdout.strip()
if not REPO_REF_OVERRIDE.strip() and RESOLVED_COMMIT != REPO_COMMIT:
    raise RuntimeError(f"Expected pinned commit {REPO_COMMIT}, got {RESOLVED_COMMIT}")
print(f"Checked out commit: {RESOLVED_COMMIT}")

## Step 2: Create Isolated Venv & Install Pinned T4 Stack

Creates `/content/kernelbench-venv` without modifying host NVIDIA drivers or system site-packages. Installs `torch==2.6.0+cu124` and `triton==3.2.0` (compatible with Tesla T4 `sm75`), along with pinned `transformers==4.57.6`, `peft==0.18.0`, and `bitsandbytes>=0.48,<0.50`.

In [ ]:
if not VENV_PYTHON.exists():
    run_cmd([sys.executable, "-m", "venv", str(VENV_DIR)])
else:
    print(f"Reusing existing virtualenv at {VENV_DIR}")

run_cmd([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], echo=False)

# Install PyTorch 2.6.0+cu124 from the official CUDA 12.4 wheel index
run_cmd(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "torch==2.6.0+cu124",
        "--index-url",
        PYTORCH_CU124_INDEX,
    ]
)

# Install Triton 3.2.0 and pinned HF/PEFT/bitsandbytes + reporting dependencies
run_cmd(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "triton==3.2.0",
        "transformers==4.57.6",
        "peft==0.18.0",
        "bitsandbytes>=0.48,<0.50",
        "accelerate",
        "numpy",
        "matplotlib",
        "jsonschema",
        "huggingface_hub",
    ]
)
print("Isolated venv installation complete.")

## Step 3: GPU Check (T4 / sm75) & Real Triton Worker Preflight

Runs **before** any 14 GB model download. Confirms CUDA visibility inside `/content/kernelbench-venv`, checks compute capability (`sm75` on Tesla T4), and grades the known `rmsnorm_2048` Triton baseline through the real isolated GPU worker (`sandbox_runner.gpu_verifier.grade_candidate`).

If `RUN_FULL_TRAINING_AND_EVAL` is `False`, execution stops cleanly after this preflight.

In [ ]:
PREFLIGHT_SCRIPT = """
import json
import os
import sys
import torch
import triton
import transformers
import peft
import bitsandbytes

from kernels.triton_baselines import TRITON_BASELINES
from sandbox_runner.gpu_verifier import grade_candidate

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available in /content/kernelbench-venv. Select a T4 GPU runtime in Colab.")

props = torch.cuda.get_device_properties(0)
gpu_name = torch.cuda.get_device_name(0)
sm_cap = f"sm{props.major}{props.minor}"
vram_gb = round(props.total_memory / (1024 ** 3), 2)

rec = grade_candidate(
    "rmsnorm_2048",
    TRITON_BASELINES["rmsnorm_2048"],
    backend="triton",
    float_dtype="fp16",
    seed=42,
    independent_seed=10042,
    warmup=2,
    iters=10,
    timeout_secs=120.0,
    run_id="preflight-rmsnorm-2048",
    method="preflight",
)
if rec.get("status") != "PASS":
    raise RuntimeError(
        f"Real GPU worker preflight FAILED: status={rec.get('status')} error={rec.get('error_message')}"
    )

payload = {
    "gpu": gpu_name,
    "sm_capability": sm_cap,
    "vram_gb": vram_gb,
    "ld_library_path_set": bool(os.environ.get("LD_LIBRARY_PATH")),
    "torch_version": torch.__version__,
    "triton_version": triton.__version__,
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
    "bitsandbytes_version": bitsandbytes.__version__,
    "preflight_status": rec["status"],
    "preflight_eager_ms": rec["eager_ms"],
    "preflight_candidate_ms": rec["candidate_ms"],
    "preflight_speedup_vs_eager": rec["speedup_vs_eager"],
}
print("PREFLIGHT_JSON:" + json.dumps(payload, allow_nan=False))
"""

preflight_env = build_subprocess_env({"PYTHONPATH": str(REPO_DIR)})
preflight_proc = run_cmd(
    [str(VENV_PYTHON), "-c", PREFLIGHT_SCRIPT],
    cwd=REPO_DIR,
    env=preflight_env,
    echo=False,
)

preflight_line = ""
for ln in preflight_proc.stdout.splitlines():
    if ln.startswith("PREFLIGHT_JSON:"):
        preflight_line = ln[len("PREFLIGHT_JSON:") :]
if not preflight_line:
    raise RuntimeError(f"Missing PREFLIGHT_JSON output. Raw stdout:\n{preflight_proc.stdout}")

PREFLIGHT_INFO: dict[str, Any] = json.loads(preflight_line)
print(json.dumps(PREFLIGHT_INFO, indent=2))
if PREFLIGHT_INFO["sm_capability"] != "sm75":
    print(f"Note: Detected {PREFLIGHT_INFO['gpu']} ({PREFLIGHT_INFO['sm_capability']}), not Tesla T4 (sm75).")

if not RUN_FULL_TRAINING_AND_EVAL:
    print(
        "\nSmoke preflight PASSED. RUN_FULL_TRAINING_AND_EVAL is False, so model download "
        "(~14 GB), training, held-out evaluation, and export will be skipped."
    )

## Step 4: Opt-In Model Snapshot Download & ExperimentConfig Setup

Runs **only** when `RUN_FULL_TRAINING_AND_EVAL = True`. Refuses to overwrite an existing run directory under `/content/t4-results`, downloads the pinned public model revision with `token=False`, and writes `config.json` using `ExperimentConfig.preset_t4_7b_qlora(...)`.

In [ ]:
CONFIG_PATH: Path = RUN_DIR / "config.json"
CONFIG_HASH: str = ""

if not RUN_FULL_TRAINING_AND_EVAL:
    print("Skipping Step 4 (smoke mode active).")
else:
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()) and not RESUME_FROM_CHECKPOINT.strip():
        raise FileExistsError(
            f"Refusing to overwrite existing non-empty run directory: {RUN_DIR}. "
            "Set a new RUN_NAME or provide RESUME_FROM_CHECKPOINT explicitly."
        )
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # Download pinned public model weights without tokens or paid endpoints
    DOWNLOAD_SCRIPT = """
import json
import sys
from huggingface_hub import snapshot_download

model_id, revision = sys.argv[1], sys.argv[2]
local_path = snapshot_download(repo_id=model_id, revision=revision, token=False,
    allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "LICENSE", "README.md"])
print("SNAPSHOT_PATH:" + json.dumps({"local_path": local_path}))
"""
    dl_proc = run_cmd(
        [str(VENV_PYTHON), "-c", DOWNLOAD_SCRIPT, MODEL_ID, MODEL_REVISION],
        cwd=REPO_DIR,
        env=build_subprocess_env({"PYTHONPATH": str(REPO_DIR)}),
        echo=False,
    )
    print(dl_proc.stdout.strip())

    # Build and save ExperimentConfig.preset_t4_7b_qlora
    CFG_SCRIPT = """
import json
import sys
from pathlib import Path
from rl_trainer.experiment_config import ExperimentConfig

params = json.loads(sys.argv[1])
out_path = Path(sys.argv[2])
cfg = ExperimentConfig.preset_t4_7b_qlora(**params)
cfg.to_json(out_path)
print("CONFIG_HASH:" + cfg.config_hash())
"""
    cfg_overrides = {
        "model_name_or_path": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_steps": MAX_STEPS,
        "group_size": GROUP_SIZE,
        "checkpoint_every": CHECKPOINT_EVERY,
        "warmup_iters": WARMUP_ITERS,
        "bench_iters": BENCH_ITERS,
        "worker_timeout_secs": WORKER_TIMEOUT_SECS,
        "checkpoint_dir": str(RUN_DIR / "checkpoints"),
        "output_jsonl": str(RUN_DIR / "train_rollouts.jsonl"),
        "resume_from": RESUME_FROM_CHECKPOINT.strip(),
    }
    cfg_proc = run_cmd(
        [str(VENV_PYTHON), "-c", CFG_SCRIPT, json.dumps(cfg_overrides), str(CONFIG_PATH)],
        cwd=REPO_DIR,
        env=build_subprocess_env({"PYTHONPATH": str(REPO_DIR)}),
        echo=False,
    )
    for ln in cfg_proc.stdout.splitlines():
        if ln.startswith("CONFIG_HASH:"):
            CONFIG_HASH = ln[len("CONFIG_HASH:") :].strip()
    if not CONFIG_HASH:
        raise RuntimeError(f"Failed to read CONFIG_HASH. Output:\n{cfg_proc.stdout}")
    print(f"Wrote {CONFIG_PATH} (config_hash={CONFIG_HASH})")

## Step 5: Run 7B QLoRA GRPO Training & Verify Optimizer Steps + Adapter SHA-256

Runs `rl_trainer.gpu_train_grpo` using the saved `config.json`, verifies that `optimizer_step_count > 0` in the saved checkpoint metadata, and computes the deterministic SHA-256 digest of the saved LoRA adapter weights.

In [ ]:
TRAIN_SUMMARY: dict[str, Any] = {}
CHECKPOINT_WEIGHTS_SHA: str = ""

if not RUN_FULL_TRAINING_AND_EVAL:
    print("Skipping Step 5 (smoke mode active).")
else:
    train_env = build_subprocess_env({"PYTHONPATH": str(REPO_DIR)})
    run_cmd(
        [str(VENV_PYTHON), "-m", "rl_trainer.gpu_train_grpo", "--config", str(CONFIG_PATH)],
        cwd=REPO_DIR,
        env=train_env,
    )

    VERIFY_CKPT_SCRIPT = """
import hashlib
import json
import sys
from pathlib import Path
from rl_trainer.checkpointing import resolve_checkpoint_dir

ckpt_root = Path(sys.argv[1])
step_dir = resolve_checkpoint_dir(ckpt_root)
state = json.loads((step_dir / "checkpoint_state.json").read_text(encoding="utf-8"))
opt_steps = int(state.get("optimizer_step_count", 0))
if opt_steps <= 0:
    raise RuntimeError(
        f"Training completed with optimizer_step_count={opt_steps}; expected > 0."
    )

hasher = hashlib.sha256()
hashed_files = []
adapter_rel = state.get("adapter_dir", "")
if adapter_rel:
    adapter_path = step_dir / adapter_rel
    for fp in sorted(p for p in adapter_path.rglob("*") if p.is_file()):
        rel = fp.relative_to(step_dir).as_posix()
        hasher.update(rel.encode("utf-8"))
        hasher.update(fp.read_bytes())
        hashed_files.append(rel)
elif state.get("model_weights_file"):
    fp = step_dir / state["model_weights_file"]
    hasher.update(fp.name.encode("utf-8"))
    hasher.update(fp.read_bytes())
    hashed_files.append(fp.name)
else:
    raise RuntimeError(f"No saved adapter_dir or model_weights_file in {step_dir}")

summary = {
    "step_dir": str(step_dir),
    "step": int(state.get("step", 0)),
    "optimizer_step_count": opt_steps,
    "total_rollouts": int(state.get("total_rollouts", 0)),
    "config_hash": state.get("config_hash", ""),
    "checkpoint_weights_sha": state["trainable_weights_sha256"],
    "adapter_artifacts_sha256": hasher.hexdigest(),
    "hashed_files": hashed_files,
}
print("CKPT_SUMMARY:" + json.dumps(summary, allow_nan=False))
"""
    ckpt_proc = run_cmd(
        [str(VENV_PYTHON), "-c", VERIFY_CKPT_SCRIPT, str(RUN_DIR / "checkpoints")],
        cwd=REPO_DIR,
        env=train_env,
        echo=False,
    )
    for ln in ckpt_proc.stdout.splitlines():
        if ln.startswith("CKPT_SUMMARY:"):
            TRAIN_SUMMARY = json.loads(ln[len("CKPT_SUMMARY:") :])
    if not TRAIN_SUMMARY:
        raise RuntimeError(f"Failed to parse checkpoint summary:\n{ckpt_proc.stdout}")

    CHECKPOINT_WEIGHTS_SHA = TRAIN_SUMMARY["checkpoint_weights_sha"]
    print(json.dumps(TRAIN_SUMMARY, indent=2))
    print(
        f"Verified 7B QLoRA adapter update: optimizer_steps={TRAIN_SUMMARY['optimizer_step_count']}, "
        f"weights_sha256={CHECKPOINT_WEIGHTS_SHA} "
        f"(7B QLoRA trains {EXPECTED_QLORA_TRAINABLE_PARAMS:,} adapter params, not full-parameter 7B)."
    )
    training_env = {**PREFLIGHT_INFO, **TRAIN_SUMMARY, "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION, "source_commit": RESOLVED_COMMIT,
        "training_mode": "qlora_7b_nf4", "dtype": "fp16",
        "config_hash": CONFIG_HASH, "training_performed": True}
    (RUN_DIR / "environment.json").write_text(json.dumps(training_env, indent=2, allow_nan=False)+"\n")


## Step 6: Held-Out Evaluation (`base_pass1`, `base_best_of_8`, `trained_checkpoint`) & Environment Manifest

Runs `rl_trainer.evaluate_heldout` with leak-free `validation_seed_regrade` selection, writes `heldout_environment.json` for `eval/compare_heldout.py`, and generates the comparison report if `eval/compare_heldout.py` is available in the checked-out commit.

In [ ]:
HELDOUT_JSONL: Path = RUN_DIR / "heldout.jsonl"
ENV_JSON: Path = RUN_DIR / "heldout_environment.json"

if not RUN_FULL_TRAINING_AND_EVAL:
    print("Skipping Step 6 (smoke mode active).")
else:
    eval_env = build_subprocess_env({"PYTHONPATH": str(REPO_DIR)})
    run_cmd(
        [
            str(VENV_PYTHON),
            "-m",
            "rl_trainer.evaluate_heldout",
            "--config",
            str(CONFIG_PATH),
            "--checkpoint-dir",
            str(RUN_DIR / "checkpoints"),
            "--selection-mode",
            "validation_seed_regrade",
            "--out",
            str(HELDOUT_JSONL),
        ],
        cwd=REPO_DIR,
        env=eval_env,
    )

    env_manifest = {
        "run_name": RUN_NAME,
        "model_id": MODEL_ID,
        "revision": MODEL_REVISION,
        "training_mode": "qlora_7b_nf4",
        "trainable_adapter_params": EXPECTED_QLORA_TRAINABLE_PARAMS,
        "dtype": "fp16",
        "gpu": PREFLIGHT_INFO["gpu"],
        "sm_capability": PREFLIGHT_INFO["sm_capability"],
        "config_hash": CONFIG_HASH,
        "source_commit": RESOLVED_COMMIT,
        "checkpoint_weights_sha": CHECKPOINT_WEIGHTS_SHA,
        "optimizer_step_count": TRAIN_SUMMARY["optimizer_step_count"],
        "torch_version": PREFLIGHT_INFO["torch_version"],
        "triton_version": PREFLIGHT_INFO["triton_version"],
        "transformers_version": PREFLIGHT_INFO["transformers_version"],
        "peft_version": PREFLIGHT_INFO["peft_version"],
        "bitsandbytes_version": PREFLIGHT_INFO["bitsandbytes_version"],
    }
    ENV_JSON.write_text(json.dumps(env_manifest, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    print(f"Wrote {ENV_JSON}")

    if (REPO_DIR / "eval" / "compare_heldout.py").is_file():
        report_dir = RUN_DIR / "report"
        run_cmd(
            [
                str(VENV_PYTHON),
                "-m",
                "eval.compare_heldout",
                "--run-dir",
                str(RUN_DIR),
                "--out",
                str(report_dir),
            ],
            cwd=REPO_DIR,
            env=eval_env,
        )
        print(f"Comparison report written to {report_dir}")

## Step 7: Export Reproducibility Bundle (Manual Download)

Packages raw JSONL records, `config.json`, `heldout_environment.json`, generated report files (if present), and the saved adapter checkpoint + optimizer + RNG state into a single ZIP under `/content/t4-results`. Run the final cell manually in Colab to trigger browser download.

In [ ]:
BUNDLE_ZIP: Path = RESULTS_ROOT / f"{RUN_NAME}_bundle.zip"

if not RUN_FULL_TRAINING_AND_EVAL:
    print("Skipping Step 7 export (smoke mode active).")
else:
    if BUNDLE_ZIP.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing bundle archive: {BUNDLE_ZIP}. Rename RUN_NAME or remove the old archive."
        )
    required_files = [
        RUN_DIR / "config.json",
        RUN_DIR / "heldout_environment.json",
        RUN_DIR / "train_rollouts.jsonl",
        RUN_DIR / "heldout.jsonl",
    ]
    for req in required_files:
        if not req.is_file():
            raise FileNotFoundError(f"Expected output artifact missing before export: {req}")

    with zipfile.ZipFile(BUNDLE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(p for p in RUN_DIR.rglob("*") if p.is_file()):
            arcname = f"{RUN_NAME}/{fp.relative_to(RUN_DIR).as_posix()}"
            zf.write(fp, arcname)

    size_mb = BUNDLE_ZIP.stat().st_size / (1024 ** 2)
    print(f"Created bundle archive: {BUNDLE_ZIP} ({size_mb:.2f} MB)")

In [ ]:
# Manual download cell: flip to True when running interactively in Google Colab.
TRIGGER_COLAB_DOWNLOAD: bool = False

if TRIGGER_COLAB_DOWNLOAD and BUNDLE_ZIP.is_file():
    from google.colab import files  # type: ignore

    files.download(str(BUNDLE_ZIP))
else:
    print(f"Manual download skipped (TRIGGER_COLAB_DOWNLOAD={TRIGGER_COLAB_DOWNLOAD}, exists={BUNDLE_ZIP.is_file()}).")